# Exercitiul 2 - Extragere informatii vizuale: Test de matematica cu puncte

**Varianta:** Imagini cu teste de matematica de forma:
```
[ • • ] + [ • • • ] = ( )
```
Fiecare patrat contine intre 1 si 6 puncte (ca pe un zar).
Trebuie sa numeri punctele din fiecare patrat si sa calculezi suma.

**Format output** (fisier `.txt` cu acelasi nume ca imaginea):
```
suma
```
Exemplu: daca ai 2 puncte + 4 puncte -> fisierul contine `6`

**Scor:** 0.45 puncte per imagine corecta

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# ============================================================
# CONFIGURARE
# ============================================================

TRAIN_DIR  = "data/ex2/antrenare"
TEST_DIR   = "data/ex2/testare"
OUTPUT_DIR = "data/ex2/output"

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# ============================================================
# INSPECTIE DATE ANTRENARE
# Vizualizeaza imaginile si raspunsurile corecte.
# ============================================================

img_files = sorted([f for f in os.listdir(TRAIN_DIR)
                    if f.endswith('.jpg') or f.endswith('.png')])

print(f"Nr imagini antrenare: {len(img_files)}")

# Afiseaza primele 3
fig, axes = plt.subplots(1, min(3, len(img_files)), figsize=(15, 5))
if len(img_files) == 1:
    axes = [axes]
for i, fname in enumerate(img_files[:3]):
    img = cv2.imread(os.path.join(TRAIN_DIR, fname))
    txt = open(os.path.join(TRAIN_DIR, os.path.splitext(fname)[0] + '.txt')).read().strip()
    axes[i].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    axes[i].set_title(f"{fname}\nRaspuns: {txt}")
    axes[i].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# PASUL 1: DETECTARE PATRATE
# Imaginea contine 2 patrate (operanzii) si un cerc (rezultatul).
# Cautam contururile dreptunghiulare.
# ============================================================

def detecteaza_patrate(img):
    """
    Returneaza lista de imagini decupate din patratele de operanzi,
    sortate de la stanga la dreapta.
    """
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)

    # Binarizare adaptativa (robusta la variatie de luminozitate)
    thresh = cv2.adaptiveThreshold(blur, 255,
                                   cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                   cv2.THRESH_BINARY_INV, 11, 2)

    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    patrate = []
    h_img, w_img = img.shape[:2]

    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area < 1000:  # ignora contururi mici
            continue
        perimeter = cv2.arcLength(cnt, True)
        approx = cv2.approxPolyDP(cnt, 0.03 * perimeter, True)

        if len(approx) == 4:  # dreptunghi/patrat
            x, y, w, h = cv2.boundingRect(approx)
            # Filtreaza: patratele de operanzi au aspect ratio ~1
            aspect = w / float(h)
            if 0.5 < aspect < 2.0 and w > 30 and h > 30:
                patrate.append((x, y, w, h))

    # Sorteaza de la stanga la dreapta
    patrate = sorted(patrate, key=lambda r: r[0])

    # Returneaza doar primele 2 (ignoram cercul rezultatului)
    crops = []
    for (x, y, w, h) in patrate[:2]:
        margin = 5
        crop = img[max(0, y+margin):y+h-margin, max(0, x+margin):x+w-margin]
        crops.append(crop)

    return crops

# Test vizual
if img_files:
    sample = cv2.imread(os.path.join(TRAIN_DIR, img_files[0]))
    crops = detecteaza_patrate(sample)
    print(f"Nr patrate detectate: {len(crops)}")
    fig, axes = plt.subplots(1, len(crops) + 1, figsize=(12, 4))
    axes[0].imshow(cv2.cvtColor(sample, cv2.COLOR_BGR2RGB))
    axes[0].set_title('Original')
    for i, crop in enumerate(crops):
        axes[i+1].imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
        axes[i+1].set_title(f'Patrat {i+1}')
    plt.show()

In [ ]:
# ============================================================
# PASUL 2: NUMARARE PUNCTE INTR-UN PATRAT
# Punctele sunt cercuri mici -> HoughCircles sau blob detection.
# ============================================================

def numara_puncte(cell_img):
    """
    Numara punctele (cercuri mici) dintr-o imagine de patrat.
    Returneaza un intreg intre 1 si 6.
    """
    if cell_img is None or cell_img.size == 0:
        return 0

    gray = cv2.cvtColor(cell_img, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 1)

    # --- Metoda 1: HoughCircles ---
    circles = cv2.HoughCircles(
        blur, cv2.HOUGH_GRADIENT, dp=1,
        minDist=10,
        param1=50, param2=15,       # <-- CALIBREAZA pe antrenare
        minRadius=3, maxRadius=25   # <-- CALIBREAZA pe antrenare
    )
    if circles is not None:
        return len(circles[0])

    # --- Metoda 2: SimpleBlobDetector (fallback) ---
    params = cv2.SimpleBlobDetector_Params()
    params.filterByArea     = True
    params.minArea          = 30
    params.maxArea          = 2000
    params.filterByCircularity = True
    params.minCircularity   = 0.5
    params.filterByConvexity = True
    params.minConvexity     = 0.7
    detector = cv2.SimpleBlobDetector_create(params)
    keypoints = detector.detect(blur)
    return len(keypoints)

# Calibrare vizuala pe antrenare
if img_files:
    for fname in img_files:
        img    = cv2.imread(os.path.join(TRAIN_DIR, fname))
        gt_txt = open(os.path.join(TRAIN_DIR, os.path.splitext(fname)[0] + '.txt')).read().strip()
        crops  = detecteaza_patrate(img)
        if len(crops) == 2:
            n1 = numara_puncte(crops[0])
            n2 = numara_puncte(crops[1])
            pred = n1 + n2
            gt   = int(gt_txt)
            ok   = '✓' if pred == gt else '✗'
            print(f"{fname}: {n1}+{n2}={pred} | GT={gt} {ok}")
        else:
            print(f"{fname}: DETECTARE ESUATA (patrate={len(crops)})")

In [ ]:
# ============================================================
# ALTERNATIVA: NUMARARE PRIN CONTURURI (daca HoughCircles nu merge)
# ============================================================

def numara_puncte_contururi(cell_img):
    """
    Alternativa la HoughCircles: cautam blob-uri negre pe fond deschis.
    """
    if cell_img is None or cell_img.size == 0:
        return 0

    gray  = cv2.cvtColor(cell_img, cv2.COLOR_BGR2GRAY)
    blur  = cv2.GaussianBlur(gray, (3, 3), 0)
    # Inversam: punctele negre devin albe
    _, thresh = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    count = 0
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area < 20 or area > 3000:  # <-- CALIBREAZA
            continue
        # Verifica circularitate
        perim = cv2.arcLength(cnt, True)
        if perim == 0:
            continue
        circ = 4 * np.pi * area / (perim ** 2)
        if circ > 0.4:  # <-- CALIBREAZA
            count += 1

    return count

In [ ]:
# ============================================================
# PASUL 3: GENERARE OUTPUT PENTRU TESTARE
# ============================================================

# ALEGE metoda care a mers mai bine pe antrenare:
# numara_puncte  (HoughCircles) sau numara_puncte_contururi
METODA_NUMARARE = numara_puncte  # <-- schimba daca e nevoie

test_files = sorted([f for f in os.listdir(TEST_DIR)
                     if f.endswith('.jpg') or f.endswith('.png')])

for fname in test_files:
    img   = cv2.imread(os.path.join(TEST_DIR, fname))
    crops = detecteaza_patrate(img)

    if len(crops) == 2:
        n1 = METODA_NUMARARE(crops[0])
        n2 = METODA_NUMARARE(crops[1])
        suma = n1 + n2
    else:
        # Fallback: daca nu s-au detectat 2 patrate, pune o valoare implicita
        suma = 0
        print(f"ATENTIE: {fname} - detectate {len(crops)} patrate")

    out_name = os.path.splitext(fname)[0] + '.txt'
    out_path = os.path.join(OUTPUT_DIR, out_name)
    with open(out_path, 'w') as f:
        f.write(str(suma) + '\n')

print(f"Output generat in: {OUTPUT_DIR}")

# Afiseaza toate predictiile
for fname in sorted(os.listdir(OUTPUT_DIR)):
    content = open(os.path.join(OUTPUT_DIR, fname)).read().strip()
    print(f"{fname}: {content}")